# Advanced Patterns

This notebook covers advanced grasp-agents patterns that build on the basics
from `agents_demo.ipynb`: multimodal input, provider-specific features, and the
full hook system. Multi-agent composition, routing, and review loops live in
`orchestration_durability_demo.ipynb`.

**Sections:**
1. Multimodal Input — images, `InputRenderable`, `Content` API
2. Provider-Specific Features — Anthropic thinking, OpenAI web search, Gemini grounding
3. Advanced Hooks — initial context builder, tool input converter, final answer extractor

---
## Setup

In [ ]:
from typing import Any, Sequence

from dotenv import load_dotenv
from pydantic import BaseModel

from grasp_agents import (
    LLMAgent,
    RunContext,
    render_events,
    BaseTool,
    InputRenderableModel,
    InputImage,
    Response,
)
from grasp_agents.types.content import InputPart, InputText, InputImage
from grasp_agents.types.events import ProcPacketOutEvent

load_dotenv()

In [ ]:
# Default LLM — swap as needed (see agents_demo.ipynb for all providers)
from grasp_agents.llm_providers.openai_responses import OpenAIResponsesLLM

llm = OpenAIResponsesLLM(model_name="gpt-5.4-nano")

---
## 1. Multimodal Input

Agents accept images alongside text. Use `InputImage` to send images from URLs,
base64 strings, or local files.

For custom input formatting, subclass `InputRenderableModel` and implement
`to_input_parts()` — the agent calls it instead of JSON-serializing the model.
(It's also a runtime-checkable `InputRenderable` protocol, so a plain `BaseModel`
with `to_input_parts()` works too.)

In [ ]:
# Simple: pass images directly as chat_inputs
agent = LLMAgent[None, str, None](
    name="vision",
    llm=llm,
    sys_prompt="Describe images in detail. Be concise."
)
out = await agent.run(
    [
        "What's in this image?",
        InputImage.from_url(
            "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
        ),
    ],
)
print(out.payloads[0])

Control how a typed input renders as LLM
input. 

Inherit it and implement to_input_parts(); the agent calls that instead
of JSON-serializing the model. 

(InputRenderable is also a runtime-checkable
Protocol, so a plain BaseModel with to_input_parts() works too — the base just
hands you the method stub and signature.)

In [ ]:
class AnalysisRequest(InputRenderableModel):
    """A typed input that controls its own LLM representation."""

    question: str
    image_url: str
    context: str = ""

    def to_input_parts(self) -> list[InputPart] | str:
        parts: list[InputPart] = []
        if self.context:
            parts.append(InputText(text=f"Context: {self.context}"))
        parts.append(InputText(text=self.question))
        parts.append(InputImage.from_url(self.image_url))

        return parts


analyst = LLMAgent[AnalysisRequest, str, None](
    name="image_analyst",
    llm=llm,
    sys_prompt="Analyze images based on the given question and context.",
    reset_transcript_on_run=True,
    stream_llm=True
)

# markdown=True renders the assistant's final answer as formatted Markdown
# (rendered once complete; thinking still streams live token-by-token).
async for event in render_events(
    analyst.run_stream(
        in_args=AnalysisRequest(
            question="What animal is this and what is it doing?",
            image_url="https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg",
            context="We're cataloging animals for a wildlife database.",
        ),
    ),
    markdown=True,
):
    pass

---
## 2. Provider-Specific Features

Each native provider exposes model-specific capabilities through its settings object.
The framework handles conversion — your agent code stays the same.

### OpenAI

In [ ]:
from grasp_agents.llm_providers.openai_responses import (
    OpenAIResponsesLLM,
    OpenAIResponsesLLMSettings,
)
from openai.types.responses import WebSearchToolParam
from openai.types.shared import Reasoning

search_llm = OpenAIResponsesLLM(
    model_name="gpt-5.4-nano",
    llm_settings=OpenAIResponsesLLMSettings(
        reasoning=Reasoning(effort="medium", summary="detailed"),
        web_search=WebSearchToolParam(type="web_search"),
    ),
)

# Web search interleaves several reasoning blocks between searches — each is
# shown once as it streams, and the final answer (with citations) renders as
# Markdown when complete.
agent = LLMAgent[None, str, None](name="searcher", llm=search_llm, stream_llm=True)
async for event in render_events(
    agent.run_stream("What happened in tech news today?"),
    show_thinking=True,
    markdown=True,
):
    pass

### Anthropic

In [ ]:
# uv sync grasp-agents[anthropic]

from anthropic.types import WebSearchTool20260209Param

from grasp_agents.llm_providers.anthropic import AnthropicLLM, AnthropicLLMSettings

# Reasoning + server-side web search: the model thinks, runs web searches, and
# reasons over the results before answering.
search_thinking_llm = AnthropicLLM(
    model_name="claude-sonnet-4-6",
    llm_settings=AnthropicLLMSettings(
        max_tokens=8000,
        thinking={"type": "adaptive", "display": "summarized"},
        web_search=WebSearchTool20260209Param(
            type="web_search_20260209", name="web_search", max_uses=3
        ),
    ),
)

agent = LLMAgent[None, str, None](
    name="researcher", llm=search_thinking_llm, stream_llm=True
)
async for event in render_events(
    agent.run_stream(
        "What are the biggest AI announcements this week, and what's the common "
        "thread between them?"
    ),
    show_thinking=True,  # stream the thinking blocks live
    markdown=True,  # render the final answer as formatted Markdown
):
    pass

### Gemini

In [ ]:
# uv sync grasp-agents[gemini]

from grasp_agents.llm_providers.gemini.gemini_llm import GeminiLLM, GeminiLLMSettings
from google.genai.types import ThinkingConfigDict, ThinkingLevel

grounded_llm = GeminiLLM(
    model_name="gemini-3.1-flash-lite",
    llm_settings=GeminiLLMSettings(
        google_search={},
        thinking_config=ThinkingConfigDict(
            include_thoughts=True, thinking_level=ThinkingLevel.MEDIUM
        )
    )
    
)

agent = LLMAgent[None, str, None](name="grounded", llm=grounded_llm, stream_llm=True)
async for event in render_events(
    agent.run_stream("What happened in tech news today?"),
    show_thinking=True,
    markdown=True,
):
    pass

### LiteLLM

In [ ]:
from grasp_agents.llm_providers.litellm import LiteLLM, LiteLLMSettings

lite_llm = LiteLLM(
    model_name="gemini/gemini-3.1-flash-lite",
    llm_settings=LiteLLMSettings(
        reasoning_effort="medium"
    ),
)

agent = LLMAgent[None, str, None](name="lite_llm", llm=lite_llm, stream_llm=True)
async for event in render_events(
    agent.run_stream("What's the probability of rolling at least one 6 in four dice throws?"),
    show_thinking=True,
    markdown=True,
):
    pass

---
## 3. Advanced Hooks

A deeper look at hooks that weren't covered in `agents_demo.ipynb`:
`@add_initial_context_builder`, `@add_tool_input_converter`, `@add_input_content_builder`,
and `@add_final_answer_extractor`.

### Initial context builder — dynamic, ephemeral leading context

In [ ]:
from grasp_agents.types.items import InputItem, InputMessageItem

# An InitialContextBuilder transforms the *ephemeral* initial context — the
# system-prompt message (composed from sys_prompt + sections) that is prepended
# to the model-facing view each turn and is NEVER stored in the transcript log.
# It receives the default initial context and returns the final one: augment it
# (as here), replace it (return your own system message), or reorder. Use it for
# always-current leading context — a user profile, the date, a
# <system-reminder> — that frames the conversation without becoming part of it.
agent = LLMAgent[None, str, None](
    name="contextual",
    llm=llm,
    sys_prompt="You are a helpful assistant. Address the user by name.",
)


@agent.add_initial_context_builder
async def add_user_profile(
    messages: list[InputItem], *, exec_id: str
) -> list[InputItem]:
    note = InputMessageItem.from_text(
        "Context: the user's name is Ada; she prefers concise answers.",
        role="user",
    )
    return [*messages, note]


out = await agent.run("Greet me and share one fun fact.")
print(out.payloads[0])


### Tool input converter — inject context into tool calls

In [ ]:
class SearchState(BaseModel):
    user_region: str = "US"
    language: str = "en"


class SearchArgs(BaseModel):
    query: str


class EnrichedSearch(BaseTool[SearchArgs, str, SearchState]):
    name: str = "search"
    description: str = "Search the web for information."

    async def _run(self, inp: SearchArgs, **kwargs: Any) -> str:  # type: ignore[override]
        # Echo the query back so the converter's injected context is visible in
        # the tool result below.
        return f"Results for: {inp.query}"


state = SearchState(user_region="EU", language="de")
ctx = RunContext[SearchState](state=state)

search_agent = LLMAgent[None, str, SearchState](
    name="search_agent",
    ctx=ctx,
    llm=llm,
    tools=[EnrichedSearch()],
    sys_prompt=(
        "Always answer by calling the `search` tool with a short query. "
        "Do not ask the user clarifying questions."
    ),
)


# A tool input converter rewrites the model's tool arguments before the tool
# runs — here it appends region/language from session state, injecting context
# into every search WITHOUT exposing those fields in the tool's schema. The
# converter receives (llm_args, exec_id); read session state via a closure (as
# here) or `search_agent.ctx`, the way hooks access context across the framework.
@search_agent.add_tool_input_converter("search")
async def enrich_search(
    llm_args: SearchArgs, *, exec_id: str | None = None
) -> SearchArgs:
    return SearchArgs(
        query=f"{llm_args.query} (region: {state.user_region}, lang: {state.language})"
    )


# The tool result panel shows the enriched query — proof the converter ran.
async for event in render_events(
    search_agent.run_stream("Find the best pizza places near me"),
    markdown=True,
):
    pass

### Final answer extractor — custom stopping logic

In [ ]:
# The loop calls the extractor after every turn. Return None to keep looping
# (the agent generates again, building on its own prior output); return a string
# to stop with it as the final answer. Here we stop only once the model wraps
# its result in <final_answer> tags, returning just the inner text — so the
# agent's output is the clean answer, with the surrounding reasoning stripped.
agent = LLMAgent[None, str, None](
    name="solver",
    llm=llm,
    max_turns=4,
    sys_prompt=(
        "Solve the problem. First show your reasoning in plain text, then give "
        "the final result wrapped in <final_answer>...</final_answer> tags."
    ),
    stream_llm=True,
)


@agent.add_final_answer_extractor
def extract_final(*, response: Response, **kwargs: Any) -> str | None:
    text = response.output_text
    if "<final_answer>" in text and "</final_answer>" in text:
        start = text.index("<final_answer>") + len("<final_answer>")
        end = text.index("</final_answer>")
        return text[start:end].strip()
    return None  # no tags yet → run another turn


answer = None
async for event in render_events(
    agent.run_stream("A train travels 90 km in 1.5 hours. What is its average speed?"),
    markdown=True,
):
    if isinstance(event, ProcPacketOutEvent):
        answer = event.data.payloads[0]

# The streamed message above includes the reasoning and the tags; the extractor
# returns only the inner value:
print(f"\nExtractor returned: {answer!r}")